# 01 EDA starter

Week 1 goal: profile the claims data and write down 5 findings.

The notebook runs on **toy data** by default (`make toy` then `make simulate`). When the real unified table exists, change `PATH` below.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
PATH = '../data/processed/toy_claims_labeled.parquet'  # swap for the real table later
df = pd.read_parquet(PATH)
df.shape

## 1. Shape, types, missing values

In [ ]:
display(df.head())
display(df.dtypes.to_frame('dtype').join(df.isna().sum().rename('n_missing')))

## 2. Date range and volume by month

In [ ]:
df['month'] = pd.to_datetime(df['from_dt']).dt.to_period('M')
print(df['from_dt'].min(), df['from_dt'].max())
ax = df.groupby('month').size().plot(figsize=(9, 3))
ax.set_title('Claims per month');

## 3. Amounts by claim type

Check skew. Money columns are almost always long-tailed, so look at log scale and quantiles.

In [ ]:
display(df.groupby('claim_type')['amount'].describe(percentiles=[.5, .9, .99]).round(0))
ax = df['amount'].apply(np.log1p).hist(bins=50, figsize=(7, 3))
ax.set_title('log(1 + amount)');

## 4. Denial rate: overall and by segment

(The label is simulated. See `docs/decisions.md` ADR-001.)

In [ ]:
print('overall denial rate:', round(df['is_denied'].mean(), 4))
display(df.groupby('claim_type')['is_denied'].agg(['mean', 'count']).round(3))
bins = pd.cut(df['days_to_file'], [-1, 30, 60, 90, 365])
display(df.groupby(bins, observed=True)['is_denied'].agg(['mean', 'count']).round(3))

## 5. Denial reasons (Pareto)

In [ ]:
reasons = df.loc[df['is_denied'] == 1, 'carc_code'].value_counts()
(reasons.cumsum() / reasons.sum()).round(3).to_frame('cumulative_share').join(reasons.rename('n'))

## 6. Provider variation

Some providers should look worse. This is the signal that later becomes a provider history feature (computed as-of date, never with future data).

In [ ]:
prov = df.groupby('provider_id')['is_denied'].agg(['mean', 'count'])
ax = prov[prov['count'] >= 30]['mean'].hist(bins=30, figsize=(7, 3))
ax.set_title('Provider denial rate');

## 7. Leakage check

Columns that must never be features: `is_denied`, `carc_code`, `denial_rule_id`, `p_denial_true`.

In [ ]:
LEAKAGE = ['is_denied', 'carc_code', 'denial_rule_id', 'p_denial_true']
FEATURE_CANDIDATES = [c for c in df.columns if c not in LEAKAGE + ['claim_id', 'month']]
FEATURE_CANDIDATES

## My 5 findings (write these in your own words)

1. 
2. 
3. 
4. 
5. 